In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

In [ ]:
load_dotenv()

In [ ]:
model = init_chat_model(
    model="gpt-4o",
    model_provider="openai",
    temperature=0,
    timeout=30,
    max_retries=3,
)

### StrOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant"), ("human", "{input}")]
)

chain = prompt | model
print(25 * "-", "Without StrOutputParser", 25 * "-")
response = chain.invoke({"input": "سلام خوبی؟"})
print(response)

print(25 * "-", "With StrOutputParser", 25 * "-")
chain = prompt | model | StrOutputParser()
response = chain.invoke({"input": "سلام خوبی؟"})
print(response)

### Sequential chain

In [ ]:
from langchain_core.runnables import RunnableLambda

# Chain 1: خلاصه
summarize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Summarize the following text in one sentence."),
        ("human", "{text}"),
    ]
)
summarize_chain = summarize_prompt | model | StrOutputParser()

# Chain 2: ترجمه خلاصه
translate_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Translate the following text to Persian. \
                Return only the translated text and nothing else.\
                Do not add explanations, notes, or extra words."),
        ("human", "{text}"),
    ]
)
translate_chain = translate_prompt | model | StrOutputParser()

# ترکیب دو chain
sequential_chain = (
    summarize_chain
    | RunnableLambda(lambda x: {"text": x})  # خروجی chain اول → ورودی chain دوم
    | translate_chain
)

long_text = """
LangChain is a framework for developing applications powered by large language models.
It provides tools and abstractions to improve the customization, accuracy, and relevancy
of the information the models generate. It includes APIs and integrations for a wide
variety of components, including vector stores, document loaders, and output parsers.
"""

result = sequential_chain.invoke({"text": long_text})
print("خلاصه فارسی:")
print(result)

### Parallel Chain

In [ ]:
from langchain_core.runnables import RunnableParallel

# Parallel: چند chain همزمان اجرا می‌شن
pros_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 pros of the given product in bullet points."),
    ("human", "{product}"),
])

cons_prompt = ChatPromptTemplate.from_messages([
    ("system", "List 3 cons of the given product in bullet points."),
    ("human", "{product}"),
])

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Write a one-line summary of the given product."),
    ("human", "{product}"),
])

# هر سه chain موازی اجرا می‌شن
parallel_chain = RunnableParallel(
    pros=(pros_prompt | model | StrOutputParser()),
    cons=(cons_prompt | model | StrOutputParser()),
    summary=(summary_prompt | model | StrOutputParser()),
)

result = parallel_chain.invoke({"product": "iPhone 15 Pro"})
print("=== Summary ===")
print(result["summary"])
print("\n=== Pros ===")
print(result["pros"])
print("\n=== Cons ===")
print(result["cons"])


### Compare Parallel

In [ ]:
import time

# ====== 3 سوال که پاسخشان دقیقاً یک کلمه است ======

# سوال 1: بله/خیر
q1_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD, just name of a city."),
    ("human", "What is the Capital of Iran?")
])

# سوال 2: یک عدد
q2_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD (a number like 'two' or 'five')."),
    ("human", "How many legs does a dog have?")
])

# سوال 3: یک اسم ساده
q3_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer with ONLY ONE WORD, just the name."),
    ("human", "What color is an apple?")
])

chain1 = q1_prompt | model | StrOutputParser()
chain2 = q2_prompt | model | StrOutputParser()
chain3 = q3_prompt | model | StrOutputParser()

# ====== 1. اجرای عادی 3 بار پشت سر هم ======
start_seq = time.time()

r1 = chain1.invoke({})
r2 = chain2.invoke({})
r3 = chain3.invoke({})

end_seq = time.time()

print("=== حالت عادی (Sequential) ===")
print(f"پاسخ‌ها: '{r1}', '{r2}', '{r3}'")
print(f"زمان: {end_seq - start_seq:.3f} ثانیه\n")

# ====== 2. اجرای موازی ======
start_par = time.time()

parallel_chain = RunnableParallel(
    ans1=chain1,
    ans2=chain2,
    ans3=chain3,
)

results = parallel_chain.invoke({})
end_par = time.time()

print("=== حالت موازی (Parallel) ===")
print(f"پاسخ‌ها: '{results['ans1']}', '{results['ans2']}', '{results['ans3']}'")
print(f"زمان: {end_par - start_par:.3f} ثانیه")
print(f"\n⚡ نسبت بهبود: {(end_seq - start_seq) / (end_par - start_par):.2f}x")

### Router Chain

In [ ]:
from typing import Literal

from pydantic import BaseModel


# Router: بر اساس موضوع، به chain مناسب route می‌کند
class RouteQuery(BaseModel):
    """تشخیص موضوع سوال"""
    topic: Literal["math", "history", "science", "other"]

router_llm = model.with_structured_output(RouteQuery)

# Chain‌های تخصصی
math_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math expert. Answer math questions step by step."),
    ("human", "{question}"),
])

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a history expert. Provide historical context."),
    ("human", "{question}"),
])

science_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a science expert. Explain scientific concepts clearly."),
    ("human", "{question}"),
])

general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{question}"),
])

def route(input_dict):
    """تشخیص موضوع و انتخاب chain مناسب"""
    route_result = router_llm.invoke(input_dict["question"])
    topic = route_result.topic

    chains = {
        "math": math_prompt | model | StrOutputParser(),
        "history": history_prompt | model | StrOutputParser(),
        "science": science_prompt | model | StrOutputParser(),
        "other": general_prompt | model | StrOutputParser(),
    }

    print(f"→ Routed to: {topic}")
    return chains[topic].invoke(input_dict)

# تست router
questions = [
    "What is the derivative of x^2?",
    "When did World War II end?",
    "What is the best programming language?",
    "How does photosynthesis work?",
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {route({'question': q})[:100]}...")
